In [4]:
# =============================================
# 나만의 용돈 기입장 프로그램
# 만든 사람: 고1
# =============================================

# 날짜 쓸 때 필요한 라이브러리
from datetime import date

# ----------------------------
# 전체 데이터 저장 공간
# ----------------------------
내역_목록 = []       # 수입/지출 내역 전부 여기 저장
카테고리_예산 = {}   # 카테고리별 예산 저장
총_예산 = 0          # 이번 달 총 예산

# ----------------------------
# 함수들
# ----------------------------

def 구분선():
    print("=" * 40)

def 예산_설정():
    global 총_예산, 카테고리_예산

    구분선()
    print("📋 초기 예산 설정")
    구분선()

    while True:
        try:
            총_예산 = int(input("이번 달 총 예산을 입력하세요 (원): "))
            if 총_예산 <= 0:
                print("❌ 0보다 큰 숫자를 입력해주세요!")
                continue
            break
        except ValueError:
            print("❌ 숫자로 다시 입력해주세요!")

    print("\n카테고리별 예산을 설정할게요.")
    print("(예: 식비, 교통비, 쇼핑, 기타 등)")
    print("카테고리 입력을 끝내려면 '완료'를 입력하세요.\n")

    while True:
        카테고리 = input("카테고리 이름: ").strip()
        if 카테고리 == "완료":
            if len(카테고리_예산) == 0:
                print("❌ 카테고리를 최소 1개는 입력해야 해요!")
                continue
            break
        if 카테고리 == "":
            print("❌ 카테고리 이름을 입력해주세요!")
            continue

        while True:
            try:
                금액 = int(input(f"'{카테고리}' 예산 (원): "))
                if 금액 < 0:
                    print("❌ 0 이상의 숫자를 입력해주세요!")
                    continue
                카테고리_예산[카테고리] = 금액
                print(f"✅ '{카테고리}' 예산 {금액:,}원 설정 완료!\n")
                break
            except ValueError:
                print("❌ 숫자로 다시 입력해주세요!")

    print(f"\n✅ 총 예산 {총_예산:,}원 설정 완료!")


def 수입_기록():
    구분선()
    print("💰 수입 기록")
    구분선()

    오늘 = date.today().strftime("%Y-%m-%d")
    날짜 = input(f"날짜 (엔터 치면 오늘 {오늘}): ").strip()
    if 날짜 == "":
        날짜 = 오늘

    내용 = input("수입 내용 (예: 용돈, 알바비): ").strip()
    if 내용 == "":
        내용 = "기타 수입"

    while True:
        try:
            금액 = int(input("금액 (원): "))
            if 금액 <= 0:
                print("❌ 0보다 큰 숫자를 입력해주세요!")
                continue
            break
        except ValueError:
            print("❌ 숫자로 다시 입력해주세요!")

    내역_목록.append({
        "종류": "수입",
        "날짜": 날짜,
        "카테고리": "수입",
        "내용": 내용,
        "금액": 금액
    })

    print(f"\n✅ 수입 {금액:,}원 기록 완료!")


def 지출_기록():
    global 카테고리_예산

    구분선()
    print("💸 지출 기록")
    구분선()

    오늘 = date.today().strftime("%Y-%m-%d")
    날짜 = input(f"날짜 (엔터 치면 오늘 {오늘}): ").strip()
    if 날짜 == "":
        날짜 = 오늘

    # 카테고리 선택
    print("\n카테고리를 선택하세요:")
    카테고리_리스트 = list(카테고리_예산.keys())
    for i, 카테고리 in enumerate(카테고리_리스트, 1):
        print(f"  {i}. {카테고리}")

    while True:
        try:
            선택 = int(input("번호 선택: "))
            if 1 <= 선택 <= len(카테고리_리스트):
                선택_카테고리 = 카테고리_리스트[선택 - 1]
                break
            else:
                print("❌ 올바른 번호를 선택해주세요!")
        except ValueError:
            print("❌ 숫자로 다시 입력해주세요!")

    내용 = input("지출 내용 (예: 편의점, 버스비): ").strip()
    if 내용 == "":
        내용 = "기타"

    while True:
        try:
            금액 = int(input("금액 (원): "))
            if 금액 <= 0:
                print("❌ 0보다 큰 숫자를 입력해주세요!")
                continue
            break
        except ValueError:
            print("❌ 숫자로 다시 입력해주세요!")

    # 잔액 부족 체크
    현재_잔액 = 잔액_계산()
    if 금액 > 현재_잔액:
        print(f"\n❌ 잔액이 부족합니다! (현재 잔액: {현재_잔액:,}원)")
        return

    내역_목록.append({
        "종류": "지출",
        "날짜": 날짜,
        "카테고리": 선택_카테고리,
        "내용": 내용,
        "금액": 금액
    })

    print(f"\n✅ '{선택_카테고리}' 지출 {금액:,}원 기록 완료!")

    # 예산 80% 초과 경고
    예산_경고_체크(선택_카테고리)


def 예산_경고_체크(카테고리):
    # 해당 카테고리 지출 합계 계산
    카테고리_지출 = sum(
        내역["금액"]
        for 내역 in 내역_목록
        if 내역["종류"] == "지출" and 내역["카테고리"] == 카테고리
    )

    설정_예산 = 카테고리_예산.get(카테고리, 0)

    if 설정_예산 > 0:
        사용_비율 = 카테고리_지출 / 설정_예산 * 100
        if 사용_비율 >= 80:
            print(f"\n⚠️  [경고] '{카테고리}' 지출이 예산의 {사용_비율:.0f}%를 넘었어!")
            print(f"   예산: {설정_예산:,}원 / 지출: {카테고리_지출:,}원")
            print("   너 돈 너무 많이 썼어! 😱")


def 잔액_계산():
    총_수입 = sum(내역["금액"] for 내역 in 내역_목록 if 내역["종류"] == "수입")
    총_지출 = sum(내역["금액"] for 내역 in 내역_목록 if 내역["종류"] == "지출")
    return 총_예산 + 총_수입 - 총_지출


def 내역_출력():
    구분선()
    print("📒 전체 내역")
    구분선()

    if len(내역_목록) == 0:
        print("아직 기록된 내역이 없어요!")
        return

    for i, 내역 in enumerate(내역_목록, 1):
        종류표시 = "💰 수입" if 내역["종류"] == "수입" else "💸 지출"
        print(f"{i}. [{내역['날짜']}] {종류표시} | {내역['카테고리']} | {내역['내용']} | {내역['금액']:,}원")

    구분선()
    잔액 = 잔액_계산()
    총_지출 = sum(내역["금액"] for 내역 in 내역_목록 if 내역["종류"] == "지출")
    총_수입 = sum(내역["금액"] for 내역 in 내역_목록 if 내역["종류"] == "수입")

    print(f"총 예산:  {총_예산:,}원")
    print(f"총 수입:  +{총_수입:,}원")
    print(f"총 지출:  -{총_지출:,}원")
    print(f"현재 잔액: {잔액:,}원")


# ----------------------------
# 메인 프로그램 시작
# ----------------------------

def 메인():
    print()
    구분선()
    print("   💵 나만의 용돈 기입장 💵")
    구분선()

    # 예산 먼저 설정
    예산_설정()

    # 메인 루프
    while True:
        print()
        구분선()
        print("📌 무엇을 할까요?")
        print("  1. 수입 기록")
        print("  2. 지출 기록")
        print("  3. 내역 보기")
        print("  4. 종료")
        구분선()

        선택 = input("번호 입력: ").strip()

        if 선택 == "1":
            수입_기록()
        elif 선택 == "2":
            지출_기록()
        elif 선택 == "3":
            내역_출력()
        elif 선택 == "4":
            구분선()
            print("프로그램을 종료합니다. 용돈 아껴써요! 👋")
            구분선()
            break
        else:
            print("❌ 1~4 중에서 입력해주세요!")

